# 04 — Messages: the Fundamental Unit of Context

Everything you've seen so far — agents, tools, streaming — moves **messages** around. This notebook zooms in on the message schema itself: the four classes, what each carries, and how they link via `tool_call_id`.

Each message carries three things:

- **Role** — encoded by the class: `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`.
- **Content** — text, or a list of content blocks (multimodal).
- **Metadata** — `response_metadata` (token usage, finish reason) and `additional_kwargs` (provider-specific extras like reasoning traces).

**Provider:** `groq:qwen/qwen3-32b`.

> Read [`04_messages.md`](./04_messages.md) alongside.


## Setup


In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
model = init_chat_model("groq:qwen/qwen3-32b")


## Two ways to call a model: string vs message list

Chat models accept either a single string (treated as one user message) or a list of message objects. Strings are fine for one-shot prompts; **always switch to messages** once you need a system prompt, conversation history, or tool exchanges.


### Way 1 — string prompt (no history, no system instructions)


In [ ]:
response = model.invoke("What is LangChain in one sentence?")
print(response.content[:400])


### Way 2 — message list with a system instruction

A `SystemMessage` sets the model's persona / constraints. It travels at the start of the message list on every invocation.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a poetry expert. Respond in 4 lines maximum."),
    HumanMessage(content="Write a poem on artificial intelligence."),
]
response = model.invoke(messages)
print(response.content)


## `SystemMessage` — prime the model's behaviour

Use it for: persona, tone, constraints, output format. The system message is the cheapest, highest-leverage knob in the API — change it and the model's whole personality shifts.


In [ ]:
expert_system = SystemMessage(content="""
You are a senior Python developer.
Always answer in <= 5 sentences and include one runnable code snippet.
""")

response = model.invoke([
    expert_system,
    HumanMessage(content="How do I create a REST API in Flask?"),
])
print(response.content[:800], '\n...')


## `HumanMessage` — user input

Represents anything sent by the user. Optional kwargs you'll occasionally use:

- `name=` — identify distinct users in a multi-user conversation.
- `id=` — your tracing identifier.
- `content=[...]` — multimodal content blocks (text + image + audio), provider-permitting.


In [ ]:
alice_msg = HumanMessage(
    content="Hello!",
    name="alice",
    id="msg_123",
)
alice_msg


## `AIMessage` — model output

Every model response is an `AIMessage`. Three slots to know:

- `.content` — the answer text. **Empty string when the model is calling a tool instead.**
- `.tool_calls` — a list of structured tool-call requests (when the model wants the application to run a function).
- `.additional_kwargs` — provider extras: Groq puts `reasoning_content` here, Gemini puts function-call signatures, etc.
- `.response_metadata` — standard-ish: token counts, finish reason, model name.


In [ ]:
response = model.invoke("What is 2 + 2?")

print("content:           ", repr(response.content[:200]))
print("tool_calls:        ", response.tool_calls)
print("additional_kwargs: ", list(response.additional_kwargs.keys()))
print("response_metadata: ", {k: response.response_metadata[k] for k in ("model_name", "finish_reason") if k in response.response_metadata})
print("usage_metadata:    ", response.usage_metadata)


## `ToolMessage` — feeding tool output back to the model

After the model emits `tool_calls`, your application runs each tool and returns the result as a `ToolMessage`. The **load-bearing detail** is `tool_call_id` — it must match the `id` of the originating call, or the model can't tell which result answers which request.


In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

# Pretend the model emitted this tool_calls request
ai_with_call = AIMessage(
    content="",
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123",
    }],
)

# Your code runs the tool and produces a result
tool_result = "Sunny, 72°F"

# Wrap it in a ToolMessage with the *matching* tool_call_id
tool_msg = ToolMessage(
    content=tool_result,
    tool_call_id="call_123",   # ← must match ai_with_call.tool_calls[0]["id"]
)

# Re-invoke the model with the full history
conversation = [
    HumanMessage(content="What's the weather in San Francisco?"),
    ai_with_call,
    tool_msg,
]
final = model.invoke(conversation)
print(final.content)


### Inspect each piece


In [ ]:
ai_with_call


In [ ]:
tool_msg


In [ ]:
final


## Provider-specific metadata: where the surprises live

Different providers tuck different extras into `additional_kwargs`:

- **Groq**: `reasoning_content` — the model's chain-of-thought before the answer.
- **Gemini**: `__gemini_function_call_thought_signatures__` — opaque signatures Gemini uses to track tool-call thoughts across turns.
- **OpenAI**: structured tool-call objects under `function_call` (legacy field) or `tool_calls`.

Treat `additional_kwargs` as a debugging surface — useful to look at, but rely on `.content`, `.tool_calls`, and `.response_metadata` (the cross-provider standard fields) in your application code.


## Recap

- Four message classes: `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`.
- The `tool_call_id` field links a tool request (on `AIMessage.tool_calls`) to its result (on `ToolMessage`).
- Use a string prompt only for one-shots; use a message list as soon as you need a system prompt, history, or tools.
- Cross-provider standard fields: `.content`, `.tool_calls`, `.response_metadata`, `.usage_metadata`.
- `.additional_kwargs` is provider-specific — useful for debugging, not for production logic.

**Course complete.** Where to go next is at the bottom of [`README.md`](./README.md).
